## Setup

In [117]:
import pandas as pd
import matplotlib.pyplot as plt

In [118]:
DATA_VERSION = 'data/2025-04-30'
CNN_DTYPE = pd.Int32Dtype()

## Intersections data

In [119]:
intersections = pd.read_csv(
    f'{DATA_VERSION}/street_intersections.csv',
    usecols=['cnn', 'data_as_of', 'st_name', 'st_type', 'the_geom'],
    dtype={'st_name': 'string', 'st_type': 'string', 'cnn': CNN_DTYPE},
    parse_dates=['data_as_of'],
)

In [120]:
intersections.dtypes

cnn                    Int32
st_name       string[python]
st_type       string[python]
the_geom              object
data_as_of    datetime64[ns]
dtype: object

In [121]:
intersections

,cnn,st_name,st_type,the_geom,data_as_of
0,51950000,GILMAN,AVE,POINT (-122.378784033 37.712929869),2025-04-30 03:33:00
1,50713000,25TH,ST,POINT (-122.380095868 37.753279508),2025-04-30 03:33:00
2,51965000,GILMAN,AVE,POINT (-122.375019661 37.710792411),2025-04-30 03:33:00
3,51943000,POLLOCK,ST,POINT (-122.365869925 37.708431923),2025-04-30 03:33:00
4,50807000,POLLOCK,ST,POINT (-122.364820405 37.709598207),2025-04-30 03:33:00
...,...,...,...,...,...
18529,23766000,SAN BRUNO,AVE,POINT (-122.404971862 37.759502499),2025-04-30 03:33:00
18530,25959000,GOLDEN GATE,AVE,POINT (-122.427016757 37.780193031),2025-04-30 03:33:00
18531,21690000,AVALON,AVE,POINT (-122.430965186 37.72692568),2025-04-30 03:33:00
18532,27317000,ANGLO,ALY,POINT (-122.474740669 37.752447862),2025-04-30 03:33:00


In [122]:
# Concatenate st_name and st_type. Note that some st_type values are missing. In that case just use st_name (do not add trailing space).
intersections['st_name'] = intersections['st_name'].str.cat(
    intersections['st_type'].str.replace('^$', '', regex=True),
    sep=' ',
    na_rep='',
)
del intersections['st_type']

In [123]:
intersections

,cnn,st_name,the_geom,data_as_of
0,51950000,GILMAN AVE,POINT (-122.378784033 37.712929869),2025-04-30 03:33:00
1,50713000,25TH ST,POINT (-122.380095868 37.753279508),2025-04-30 03:33:00
2,51965000,GILMAN AVE,POINT (-122.375019661 37.710792411),2025-04-30 03:33:00
3,51943000,POLLOCK ST,POINT (-122.365869925 37.708431923),2025-04-30 03:33:00
4,50807000,POLLOCK ST,POINT (-122.364820405 37.709598207),2025-04-30 03:33:00
...,...,...,...,...
18529,23766000,SAN BRUNO AVE,POINT (-122.404971862 37.759502499),2025-04-30 03:33:00
18530,25959000,GOLDEN GATE AVE,POINT (-122.427016757 37.780193031),2025-04-30 03:33:00
18531,21690000,AVALON AVE,POINT (-122.430965186 37.72692568),2025-04-30 03:33:00
18532,27317000,ANGLO ALY,POINT (-122.474740669 37.752447862),2025-04-30 03:33:00


In [124]:
# Join by cnn. Aggregate st_name_type by concatenating into comma separted string and take any value for the_tom, and data_as_of
# Ensure that st_name is sorted alphabetically before concatenation
intersections = intersections.groupby('cnn').agg(
    st_name_type=('st_name', lambda x: ', '.join(sorted(x))),
    the_geom=('the_geom', 'first'),
    data_as_of=('data_as_of', 'first'),
).reset_index()
intersections

,cnn,st_name_type,the_geom,data_as_of
0,20010000,"GILROY ST, JAMESTOWN AVE",POINT (-122.389437676 37.715557621),2025-04-30 03:33:00
1,20011000,"GIANTS DR, INGERSON AVE",POINT (-122.38813051 37.716298216),2025-04-30 03:33:00
2,20013000,"GILROY ST, IGNACIO AVE",POINT (-122.388977803 37.716086986),2025-04-30 03:33:00
3,20034000,"CLEO RAND AVE, DONAHUE ST",POINT (-122.370352351 37.728568176),2025-04-30 03:33:00
4,20039000,"EARL ST, JERROLD AVE",POINT (-122.372583555 37.729254505),2025-04-30 03:33:00
...,...,...,...,...
9706,54419000,WEST ACCESS,POINT (-122.421729355 37.712136371),2025-04-30 03:33:00
9707,54422000,GATES ST,POINT (-122.414403423 37.733790443),2025-04-30 03:33:00
9708,54423000,TREASURE ISLAND RD,POINT (-122.371387279 37.812108987),2025-04-30 03:33:00
9709,54424000,SIGNAL RD,POINT (-122.365802159 37.809918013),2025-04-30 03:33:00


## Streets data

In [146]:
ADDRESS_TYPE = pd.Int32Dtype()
INT_ENUM_TYPE = pd.Int8Dtype()
streets = pd.read_csv(
    f'{DATA_VERSION}/streets.csv',
    usecols=[
        'cnn',
        'street',
        'st_type',
        'lf_fadd',
        'lf_toadd',
        'rt_fadd',
        'rt_toadd',
        'f_node_cnn',
        't_node_cnn',
        'classcode',
        'accepted',
        'active',
        'oneway',
        'line',
        'data_as_of',
    ],
    dtype={
        'cnn': CNN_DTYPE,
        'lf_fadd': ADDRESS_TYPE,
        'lf_toadd': ADDRESS_TYPE,
        'rt_fadd': ADDRESS_TYPE,
        'rt_toadd': ADDRESS_TYPE,
        'f_node_cnn': CNN_DTYPE,
        't_node_cnn': CNN_DTYPE,
        'classcode': INT_ENUM_TYPE,
    },
    parse_dates=['data_as_of'],
)
streets

,cnn,lf_fadd,lf_toadd,rt_fadd,rt_toadd,street,st_type,f_node_cnn,t_node_cnn,accepted,active,classcode,oneway,line,data_as_of
0,9999000,301,399,300,398,ORTEGA,ST,27098000,27113000,True,True,5,B,"LINESTRING (-122.466600499 37.752803078, -122....",2025-04-30 03:48:00
1,10000202,501,599,0,0,ORTEGA,ST,32870000,32871000,True,True,5,T,"LINESTRING (-122.468335072 37.753243731, -122....",2025-04-30 03:48:00
2,10002000,801,899,800,898,ORTEGA,ST,27308000,27310000,True,True,5,B,"LINESTRING (-122.472007575 37.752569818, -122....",2025-04-30 03:48:00
3,10010000,1501,1599,1500,1598,ORTEGA,ST,27338000,27379000,True,True,5,B,"LINESTRING (-122.479557564 37.752234616, -122....",2025-04-30 03:48:00
4,10019000,2401,2499,2400,2498,ORTEGA,ST,27674000,27677000,True,True,5,B,"LINESTRING (-122.489193569 37.751806869, -122....",2025-04-30 03:48:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17140,14304000,0,0,0,0,UNNAMED,ST,35136000,35137000,False,True,5,B,"LINESTRING (-122.371117962 37.830712109, -122....",2025-04-30 03:48:00
17141,6535000,0,0,0,0,GRIFFITH,ST,51835000,51834000,False,True,0,B,"LINESTRING (-122.373499523 37.734495279, -122....",2025-04-30 03:48:00
17142,6150201,0,0,1876,1932,GENEVA,AVE,21359000,20428000,True,True,3,F,"LINESTRING (-122.423353436 37.709250951, -122....",2025-04-30 03:48:00
17143,6179000,0,0,0,0,GILMAN,AVE,51950000,51953000,False,True,0,B,"LINESTRING (-122.378784033 37.712929869, -122....",2025-04-30 03:48:00


In [147]:
streets.dtypes

cnn                    Int32
lf_fadd                Int32
lf_toadd               Int32
rt_fadd                Int32
rt_toadd               Int32
street                object
st_type               object
f_node_cnn             Int32
t_node_cnn             Int32
accepted                bool
active                  bool
classcode               Int8
oneway                object
line                  object
data_as_of    datetime64[ns]
dtype: object

## Speed limits

In [ ]:
speed_limits = pd.read_csv(
    f'{DATA_VERSION}/speed_limits.csv',
    usecols=['cnn', 'speedlimit', 'schoolzone', 'schoolzone_limit', 'data_as_of'],
    dtype={
        'cnn': CNN_DTYPE,
        'speedlimit': pd.Int16Dtype(),
    },
    parse_dates=['data_as_of'],
)
speed_limits

/var/folders/yb/f9_t70mn0x95vlx6y2jnbpcr0000gn/T/ipykernel_94078/181972844.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  speed_limits = pd.read_csv(


,cnn,speedlimit,schoolzone,schoolzone_limit,data_as_of
0,5903000,0,NaN,0,2016-05-10
1,10270201,35,NaN,0,2016-05-10
2,5127000,0,NaN,0,2016-05-10
3,2323000,0,NaN,0,2016-05-10
4,7534000,25,NaN,0,2016-05-10
...,...,...,...,...,...
15724,6785000,0,NaN,0,2016-05-10
15725,6533000,0,NaN,0,2016-05-10
15726,7191000,0,NaN,0,2016-05-10
15727,5575000,0,NaN,0,2016-05-10


In [149]:
speed_limits.dtypes

cnn                          Int32
speedlimit                   Int16
schoolzone                  object
schoolzone_limit             int64
data_as_of          datetime64[ns]
dtype: object

In [150]:
# Check count by cnn ID
streets['cnn'].value_counts()

cnn
9999000     1
2188000     1
2109000     1
2126000     1
2127000     1
           ..
10693000    1
10712000    1
10724000    1
10727000    1
6785000     1
Name: count, Length: 17145, dtype: Int64

In [151]:
# Join speedlimit onto streets. Only add speedlimit column
streets_v2 = streets.merge(
    speed_limits[['cnn', 'speedlimit']],
    on='cnn',
    how='left',
)
streets_v2

,cnn,lf_fadd,lf_toadd,rt_fadd,rt_toadd,street,st_type,f_node_cnn,t_node_cnn,accepted,active,classcode,oneway,line,data_as_of,speedlimit
0,9999000,301,399,300,398,ORTEGA,ST,27098000,27113000,True,True,5,B,"LINESTRING (-122.466600499 37.752803078, -122....",2025-04-30 03:48:00,0
1,10000202,501,599,0,0,ORTEGA,ST,32870000,32871000,True,True,5,T,"LINESTRING (-122.468335072 37.753243731, -122....",2025-04-30 03:48:00,0
2,10002000,801,899,800,898,ORTEGA,ST,27308000,27310000,True,True,5,B,"LINESTRING (-122.472007575 37.752569818, -122....",2025-04-30 03:48:00,0
3,10010000,1501,1599,1500,1598,ORTEGA,ST,27338000,27379000,True,True,5,B,"LINESTRING (-122.479557564 37.752234616, -122....",2025-04-30 03:48:00,0
4,10019000,2401,2499,2400,2498,ORTEGA,ST,27674000,27677000,True,True,5,B,"LINESTRING (-122.489193569 37.751806869, -122....",2025-04-30 03:48:00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17143,14304000,0,0,0,0,UNNAMED,ST,35136000,35137000,False,True,5,B,"LINESTRING (-122.371117962 37.830712109, -122....",2025-04-30 03:48:00,<NA>
17144,6535000,0,0,0,0,GRIFFITH,ST,51835000,51834000,False,True,0,B,"LINESTRING (-122.373499523 37.734495279, -122....",2025-04-30 03:48:00,0
17145,6150201,0,0,1876,1932,GENEVA,AVE,21359000,20428000,True,True,3,F,"LINESTRING (-122.423353436 37.709250951, -122....",2025-04-30 03:48:00,30
17146,6179000,0,0,0,0,GILMAN,AVE,51950000,51953000,False,True,0,B,"LINESTRING (-122.378784033 37.712929869, -122....",2025-04-30 03:48:00,<NA>


In [152]:
# Count streets by unique speed limit values descending
streets_v2['speedlimit'].value_counts()

speedlimit
0     11840
25     1847
20      910
30      543
35      271
99      233
40       44
45       24
15        4
Name: count, dtype: Int64

In [153]:
# Repalce 0 values with 25mph
streets_v2['speedlimit'] = streets_v2['speedlimit'].replace(0, 25)
streets_v2['speedlimit'].value_counts()

speedlimit
25    13687
20      910
30      543
35      271
99      233
40       44
45       24
15        4
Name: count, dtype: Int64

In [156]:
# Search streets by name mariposa
streets_v2[streets_v2['street'].str.contains('mariposa', case=False)]

,cnn,lf_fadd,lf_toadd,rt_fadd,rt_toadd,street,st_type,f_node_cnn,t_node_cnn,accepted,active,classcode,oneway,line,data_as_of,speedlimit
1768,8716002,951,999,950,998,MARIPOSA,ST,52249000,23655000,False,True,4,B,"LINESTRING (-122.393173807 37.764055599, -122....",2025-04-30 03:48:00,25
1769,8719000,1201,1299,1200,1298,MARIPOSA,ST,23676000,23680000,True,True,4,B,"LINESTRING (-122.395682317 37.763906986, -122....",2025-04-30 03:48:00,25
1770,8720000,1301,1399,1300,1398,MARIPOSA,ST,23680000,23682000,True,True,4,B,"LINESTRING (-122.396644515 37.763850619, -122....",2025-04-30 03:48:00,25
1771,8723000,1701,1799,1700,1798,MARIPOSA,ST,23749000,23752000,True,True,4,B,"LINESTRING (-122.40051424 37.763619149, -122.4...",2025-04-30 03:48:00,25
2003,8716001,901,949,900,948,MARIPOSA,ST,52248000,52249000,False,True,4,B,"LINESTRING (-122.392485618 37.764097998, -122....",2025-04-30 03:48:00,25
2004,8724000,1801,1899,1800,1898,MARIPOSA,ST,23752000,23770000,True,True,4,B,"LINESTRING (-122.401478944 37.763561459, -122....",2025-04-30 03:48:00,25
2005,8730000,2401,2499,2400,2498,MARIPOSA,ST,24019000,24021000,True,True,5,B,"LINESTRING (-122.407304847 37.763187802, -122....",2025-04-30 03:48:00,25
2006,8733000,2701,2799,2700,2798,MARIPOSA,ST,24024000,24046000,True,True,5,B,"LINESTRING (-122.410244115 37.763029333, -122....",2025-04-30 03:48:00,25
2007,8734000,2801,2899,2800,2898,MARIPOSA,ST,24046000,24045000,True,True,5,B,"LINESTRING (-122.411206839 37.762972727, -122....",2025-04-30 03:48:00,25
2008,8735000,2901,2999,2900,2998,MARIPOSA,ST,24045000,24052000,True,True,5,B,"LINESTRING (-122.412173289 37.762915894, -122....",2025-04-30 03:48:00,25


In [161]:
# Count segments by unique street, st_type pair, along with average speed, sorted by count. Show top 20 rows
streets_v2.groupby(['street', 'st_type']).agg(
    count=('cnn', 'count'),
    avg_speed=('speedlimit', 'mean'),
).sort_values('count', ascending=False).head(20)

,,count,avg_speed
street,st_type,,
GEARY,BLVD,147,26.319444
03RD,ST,142,24.795082
MISSION,ST,138,21.576923
MARKET,ST,118,24.318182
OCEAN,AVE,112,23.348214
LINCOLN,WAY,99,29.79798
ALEMANY,BLVD,91,31.724138
CALIFORNIA,ST,86,25.0
VAN NESS,AVE,84,25.0


## Injuries

In [ ]:
injuries = pd.read_csv(
    f'{DATA_VERSION}/traffic_crashes_resulting_in_injury.csv',
    # usecols=['cnn', 'speedlimit', 'schoolzone', 'schoolzone_limit', 'data_as_of'],
    # dtype={
    #     'cnn': CNN_DTYPE,
    #     'speedlimit': pd.Int16Dtype(),
    # },
    # parse_dates=['data_as_of'],
)
injuries